# 01 | k-nearest neighbors from first principles

**Question:** Can a classifier written with NumPy reproduce an established machine-learning library?

This notebook is an AI-assisted portfolio extension of ECE 657 coursework, not an original submitted assignment. Run all cells from top to bottom after installing the repository requirements.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'projects').is_dir(): root = root.parent
sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt

## 1. Algorithm
For a query $q$, compute $d_i=\\|x_i-q\\|_2$, select the $k$ smallest distances, then return the majority class. Feature scaling matters because distances depend on units.

In [ ]:
from projects.fundamentals import knn
import inspect
print(inspect.getsource(knn))

## 2. Benchmark with independent test data
Fit the scaler only on training observations. Compare identical predictions against scikit-learn, and report both accuracy and balanced accuracy.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay
X,y = load_breast_cancer(return_X_y=True)
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
scaler = StandardScaler().fit(Xtr)
Xtr,Xte = scaler.transform(Xtr),scaler.transform(Xte)
pred = np.array([knn(Xtr,ytr,q,5) for q in Xte])
ref = KNeighborsClassifier(n_neighbors=5).fit(Xtr,ytr).predict(Xte)
print({'accuracy':accuracy_score(yte,pred),'balanced_accuracy':balanced_accuracy_score(yte,pred),'agreement':np.mean(pred==ref)})
ConfusionMatrixDisplay.from_predictions(yte,pred); plt.show()

## 3. Interpretation
Recorded run: 97.9% accuracy, 97.2% balanced accuracy and identical predictions to scikit-learn on 143 held-out points. This checks algorithmic agreement, not clinical readiness. The method has inference cost proportional to the number of training samples and input dimensions.

**Practice:** Compare odd values of $k$ using training-only cross-validation, then evaluate the selected value once on a held-out set.